# Stage 2 – Molecular Dynamics Trajectory Analysis

Interactive analysis of MD simulations for wild-type SCN1A/SCN2A and carbamazepine-bound systems.

## Sections
1. Load trajectory results
2. RMSD over time
3. Per-residue RMSF
4. Secondary structure analysis
5. Carbamazepine binding contacts

**Prerequisites** – run stage2 scripts first:
```
python stage2_md_simulations/01_prepare_system.py --pdb <structure.pdb>
python stage2_md_simulations/02_run_md_simulation.py --prepared_dir <dir>
python stage2_md_simulations/03_analyze_trajectory.py --traj <traj.dcd>
python stage2_md_simulations/04_drug_binding_analysis.py --traj <traj.dcd>
```

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../..'))
from utils.visualization import plot_rmsd, plot_rmsf

RESULTS_DIR = '../../data/results/stage2'
print('Imports OK')

## 2. RMSD Over Time

Root Mean Square Deviation of Cα atoms relative to the initial (minimized) structure.

In [ ]:
rmsd_path = os.path.join(RESULTS_DIR, 'rmsd.csv')

if os.path.exists(rmsd_path):
    df_rmsd = pd.read_csv(rmsd_path)
else:
    print('RMSD CSV not found — using mock data')
    rng = np.random.default_rng(0)
    n = 500
    time = np.linspace(0, 100, n)
    rmsd_wt = np.cumsum(rng.normal(0, 0.01, n)) + 1.5 + rng.normal(0, 0.1, n)
    rmsd_wt = np.clip(rmsd_wt, 0.5, 4)
    rmsd_cbz = rmsd_wt + rng.normal(0, 0.15, n)
    df_rmsd = pd.DataFrame({'time_ns': time, 'rmsd_wt': rmsd_wt, 'rmsd_cbz': rmsd_cbz})

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_rmsd['time_ns'], df_rmsd.get('rmsd_wt', df_rmsd.iloc[:, 1]),
        lw=0.8, label='Wild-type', color='#1565C0')
if 'rmsd_cbz' in df_rmsd.columns:
    ax.plot(df_rmsd['time_ns'], df_rmsd['rmsd_cbz'],
            lw=0.8, label='+Carbamazepine', color='#E53935')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('RMSD (Å)')
ax.set_title('Cα RMSD over MD simulation')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Per-Residue RMSF

High RMSF indicates flexible regions (loops, termini). Transmembrane segments are typically rigid.

In [ ]:
rmsf_path = os.path.join(RESULTS_DIR, 'rmsf.csv')

if os.path.exists(rmsf_path):
    df_rmsf = pd.read_csv(rmsf_path)
    residues = df_rmsf['residue'].values
    rmsf = df_rmsf['rmsf'].values
else:
    print('RMSF CSV not found — using mock data')
    rng = np.random.default_rng(1)
    residues = np.arange(1, 2010)
    rmsf = np.abs(rng.normal(1.5, 0.8, len(residues)))
    # Simulate high-flexibility N/C termini
    rmsf[:50] += 2
    rmsf[-50:] += 2

# Approximate SCN1A transmembrane domain boundaries
tm_regions = [
    (121, 141, 'DI-S1'), (144, 164, 'DI-S2'), (168, 188, 'DI-S3'),
    (190, 213, 'DI-S4'), (215, 241, 'DI-S5'), (243, 262, 'DI-S6'),
]

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(residues, rmsf, lw=0.6, color='#1565C0')
ax.fill_between(residues, rmsf, alpha=0.15, color='#1565C0')
colors = plt.cm.Pastel1.colors
for i, (s, e, lbl) in enumerate(tm_regions):
    ax.axvspan(s, e, alpha=0.3, color=colors[i % len(colors)], label=lbl)
ax.set_xlabel('Residue')
ax.set_ylabel('RMSF (Å)')
ax.set_title('Per-residue RMSF — SCN1A')
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## 5. Carbamazepine Binding Analysis

In [ ]:
drug_path = os.path.join(RESULTS_DIR, 'drug_binding.csv')

if os.path.exists(drug_path):
    df_drug = pd.read_csv(drug_path)
    print('Drug binding contacts loaded:')
    display(df_drug.head(20))
else:
    print('Drug binding CSV not found — run stage2/04_drug_binding_analysis.py first')
    # Mock contact frequency
    df_drug = pd.DataFrame({
        'residue': [1458, 1462, 1466, 1591, 1595, 1599],
        'res_name': ['ILE', 'PHE', 'TYR', 'ASN', 'PHE', 'ILE'],
        'contact_freq': [0.92, 0.88, 0.75, 0.70, 0.65, 0.60],
    })
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(
        [f"{r['res_name']}{r['residue']}" for _, r in df_drug.iterrows()],
        df_drug['contact_freq'],
        color='#E53935'
    )
    ax.set_xlabel('Contact frequency')
    ax.set_title('Carbamazepine binding contacts (mock)')
    plt.tight_layout()
    plt.show()